In [ ]:
import pandas as pd
import numpy as np

In [ ]:
%matplotlib widget

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
df_L1 = pd.read_csv('halo/L1_halo.csv')
df_L2 = pd.read_csv('halo/L2_halo.csv')

In [ ]:
df_L1.head()

In [ ]:
x1 = df_L1['x']
x3 = df_L1['z']
v2 = df_L1['vy']
T = df_L1['T']

x1_2 = df_L2['x']
x3_2 = df_L2['z']
v2_2 = df_L2['vy']
T_2 = df_L2['T']

mu = 2.447832293839728e-06;

R_Venus = 6051.8 #км

In [ ]:
venus_pos = np.array([1-mu, 0, 0])

In [ ]:
from scipy.integrate import solve_ivp
import math

def odeCR3BP_3D(t, p, mu):
    # p: [x, y, z, vx, vy, vz]
    x, y, z, vx, vy, vz = p

    R1 = np.sqrt((x + mu)**2 + y**2 + z**2)
    R2 = np.sqrt((x - 1 + mu)**2 + y**2 + z**2)

    ddx = 2*vy + x - (1 - mu)*(x + mu)/R1**3 - mu*(x - 1 + mu)/R2**3
    ddy = -2*vx + y - (1 - mu)*y/R1**3 - mu*y/R2**3
    ddz = - (1 - mu)*z/R1**3 - mu*z/R2**3

    dpdt = np.array([vx, vy, vz, ddx, ddy, ddz])
    return dpdt

In [ ]:
fig_man, ax_man = plt.subplots(1, subplot_kw={"projection": "3d"})
fig_res, ax_res = plt.subplots(1)
res_list = []
for i in range(0, len(x1), 100):
    #print(i)
    p0 = np.array([x1[i], 0, x3[i], 0, v2[i], 0])
    p = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                  t_span=(0, T[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T[i], 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], p.y[2,:], 'r')
    res = np.linalg.norm(p.y[:,-1] - p0)
    ax_res.plot(x1[i], res, '.k')
    res_list.append(res)
ax_man.scatter(venus_pos[0], venus_pos[1], venus_pos[2])
ax_man.axis('equal')
plt.grid()
plt.show()

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp
import math

def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt


x1_fun = lambda x: x
x2_fun = lambda x: 0

def x3_fun(x):
    #y = (0.0024391736*math.exp(math.atan(49.569262932819*math.sin(0.5897607/math.log(x))))**0.020178359/math.sin(206.5387397382*x) + 0.014046774) #1.9e-04
    #if x>=0.9896064931 and x<0.9923756214:
    #    y = (2.39278338445475*x - 2.36231200588874 - 0.000604282058146885/math.sin(879.7134/x)) #1.6e-04
    #else: 
    #   y = (0.013622138 - 0.0021129271/math.sin(216.04132*x)) #3.0e-04
    
    #y = (0.013622138 - 0.0021129271/math.sin(216.04132*x)) #3.0e-04
    #y = (-2.11109216958297e-15*x**(-2697.0463) - 9.061731e-6*x**(-634.6433) + 0.011626389) #2.9e-04
    #y = (2.39278338445475*x - 2.36231200588874 - 0.000604282058146885/math.sin(879.7134/x)) #1.6e-04
    y = (-0.2168235 - 0.0010825446/math.cos(x**(-38.041058)) - 6.9732545e-7/(math.cos(x**(-42.989643))*math.cos(x**(-38.041058))) + 0.23019466/x) #1.5e-04
    y = (((math.log(x) + 0.011358267)*math.log(math.atan(math.exp(x**76.86457))) - 1.771093e-6)/(math.log(math.atan(math.exp(x**76.86457)))*math.sin(x*(1.3414929*x**155.64879 + 6.2817144)))) #1.1e-04
    return y

v1_fun = lambda x: 0

def v2_fun(x):
    #y = ((2.68151637513178 - 2.6779678*x)*math.atan(math.atan(math.sin(235.22978*x)))) #3.2e-04
    #y = ((0.06056452 - 0.08064385**x)*math.atan(0.66398126*math.sin(494.23343*x)) + 0.026671875*math.sin(173.20998*x**2) - 0.0020823614) #2.5e-04
    #y = (math.log(x**(-2.2831533))*math.sin(1.123034*x**(-93.746704))/math.sin(0.8404947*x**(-111.786415))) #1.1e-04
    y = (((-246.41634365676*x + 313.749*math.atan(x**1.5511825))*(7559.7954*x - 7558.7954*x**x) - 0.0021235233862764)/(7559.7954*x - 7558.7954*x**x)) #9.1e-05
    return y
    
v3_fun = lambda x: 0

def T_fun(x):
    y = (math.sin(1.3610153*math.atan(1.41910943730896*math.log(x**(-144.90504)) - 1.56562070810278) + 0.59557784) + 2.0933857) #4.5e-03
    return y
x_range = np.linspace(0.9896064931,0.9990056214, 100)
fig_man, ax_man = plt.subplots(1, subplot_kw={"projection": "3d"})
fig_res, ax_res = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), x3_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i]), v3_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:])
    ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
#ax_man.axis('equal')
plt.grid()
plt.show()

In [ ]:
plt.plot(x1, v2)
plt.plot(x1, [v2_fun(x) for x in x1],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v2')
#plt.legend()
plt.show()

In [ ]:
plt.plot(x1, x3)
plt.plot(x1, [x3_fun(x) for x in x1],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('x3')
#plt.legend()
plt.show()

In [ ]:
plt.plot(x1, T)
plt.plot(x1, [T_fun(x) for x in x1],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('T')
#plt.legend()
plt.show()

In [ ]:


fig_man, ax_man = plt.subplots(1, subplot_kw={"projection": "3d"})
fig_res, ax_res = plt.subplots(1)
res_list = []
for i in range(0, len(x1_2), 200):
    #print(i)
    p0 = np.array([x1_2[i], 0, x3_2[i], 0, v2_2[i], 0])
    p = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                  t_span=(0, T_2[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_2[i], 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], p.y[2,:])
    res = np.linalg.norm(p.y[:,-1] - p0)
    ax_res.plot(x1_2[i], res, '.k')
    res_list.append(res)
ax_man.axis('equal')
plt.grid()
plt.show()

In [ ]:
fig_man, ax_man = plt.subplots(1, subplot_kw={"projection": "3d"})
fig_res, ax_res = plt.subplots(1)
res_list = []
for i in range(0, len(x1), 100):
    #print(i)
    p0 = np.array([x1[i], 0, x3[i], 0, v2[i], 0])
    p = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                  t_span=(0, T[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T[i], 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], p.y[2,:], 'r')
    res = np.linalg.norm(p.y[:,-1] - p0)
    ax_res.plot(x1[i], res, '.k')
    res_list.append(res)
for i in range(0, len(x1_2), 200):
    #print(i)
    p0 = np.array([x1_2[i], 0, -x3_2[i], 0, v2_2[i], 0])
    p = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                  t_span=(0, T_2[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_2[i], 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], p.y[2,:], 'k')
    res = np.linalg.norm(p.y[:,-1] - p0)
    ax_res.plot(x1_2[i], res, '.k')
ax_man.scatter(venus_pos[0], venus_pos[1], venus_pos[2])
ax_man.axis('equal')
plt.grid()
plt.show()

In [ ]:
plt.plot(x1, T)
plt.plot(x1_2, T_2,'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('T')
#plt.legend()
plt.show()

In [ ]:
import math

G = 6.67430e-11          # м^3 / кг / с^2

def cr3bp_units_from_mu(mu, M_total, a_m=None, period_s=None):
    """
    Скалирующие единицы CR3BP -> СИ.

    mu        – безразмерный параметр m2/(m1+m2)
    M_total   – m1 + m2, кг
    a_m       – расстояние между телами, м   (задайте a_m  ИЛИ  period_s)
    period_s  – орбитальный период системы, с
    -----------------------------------------------
    Возврат: dict {DU, TU, VU, m1, m2}
    """
    if (a_m is None) == (period_s is None):
        raise ValueError("Задайте либо a_m, либо period_s, но не оба одновременно.")

    m2 = mu * M_total
    m1 = M_total - m2

    if period_s is not None:
        n = 2*math.pi/period_s          # рад/с
        a_m = (G*M_total/n**2)**(1/3)   # из третьего закона Кеплера
    else:
        n = math.sqrt(G*M_total/a_m**3) # рад/с (=1/TU)

    DU = a_m
    TU = 1./n                           # то же, что √(a^3/(GM))
    VU = DU/TU                          # м/с

    return {"DU": DU, "TU": TU, "VU": VU, "m1": m1, "m2": m2}

mu = 2.447832293839728e-06;
M_sun = 1.98892e30
M_venus = 4.867e24
M_sum   = M_sun + M_venus
a_Venus = 108208930 *1e3                 # м

units = cr3bp_units_from_mu(mu, M_sum, a_m=a_Venus)
print(units)

In [ ]:
#отбираем орбиты с одинаковым периодом
T_2_limits = (np.min(T_2), np.max(T_2))

Mean_list = []
Max_list = []
Min_list = []
for i in range(len(x1)):
#i = 100
    x1_i = x1[i]
    x3_i = x3[i]
    v2_i = v2[i]
    T_i = T[i]
    
    T_exists = T_i>=T_2_limits[0] and T_i<=T_2_limits[1]
    
    if T_exists:
        T_target = T_i
        x1_target = np.interp(T_i, T_2[::-1], x1_2[::-1])
        x3_target = np.interp(x1_target, x1_2[::-1], x3_2[::-1])
        v2_target = np.interp(x1_target, x1_2[::-1], v2_2[::-1])
    
        p0 = np.array([x1_i, 0, x3_i, 0, v2_i, 0])
        p = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                      t_span=(0, T[i]),
                      y0=p0,
                      method='LSODA',
                      t_eval = np.linspace(0, T_i, 100),
                      atol=1e-12,
                      rtol=1e-12)
        traj1 = p.y[:3, :]   # (3, N)
        
        
        p0 = np.array([x1_target, 0, -x3_target, 0, v2_target, 0])
        p = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                      t_span=(0, T[i]),
                      y0=p0,
                      method='LSODA',
                      t_eval = np.linspace(0, T_target, 100),
                      atol=1e-12,
                      rtol=1e-12)
        traj2 = p.y[:3, :]   # (3, N)
    
        A = traj1.T   # (N, 3)
        B = traj2.T   # (N, 3)
        P = venus_pos # (3,)
        AB = B - A
        AP = A - P
        cross = np.cross(AB, AP)
        dists = np.linalg.norm(cross, axis=1) / np.linalg.norm(AB, axis=1)
        M = np.mean(dists)
        Max = np.max(dists)
        Min = np.min(dists)
        Mean_list.append(M)
        Max_list.append(Max)
        Min_list.append(Min)
    else:
        print('No point')



In [ ]:
X

In [ ]:
R_Venus

In [ ]:
import matplotlib.patches 

In [ ]:
X[750]

In [ ]:
#plt.plot(x1, np.array(Mean_list)*units['DU']/(R_Venus*1000))
fig, ax = plt.subplots(1)
DU_local = units['DU']/(R_Venus*1000)
rect = matplotlib.patches.Rectangle((580000/R_Venus, 0), 50000/R_Venus, 10, facecolor='r', alpha=0.5,label='Линейно устойчивые гало-орбиты')
#plt.plot(2*[600000/R_Venus], [0, 10],label='Линейно устойчивые гало-орбиты')
#ax.add_patch(rect)
X = np.array((venus_pos[0]-x1)*DU_local)
plt.plot(X, np.array(Max_list)*DU_local, '-.',label='Максимальное расстояние')
plt.plot(X, np.array(Min_list)*DU_local, '--',label='Минимальное расстояние')
plt.plot([X[0], X[-1]], 2*[1],label='Поверхность Венеры')
plt.grid()
plt.legend()
plt.ylabel('Расстояние от линии измерений до\n центра Венеры, радиусы Венеры')
plt.xlabel('Наиболее дальняя точка по X для первой орбиты, радиусы Венеры')

In [ ]:
Mean_list[1203]

In [ ]:
index_min = np.argmin(Mean_list)
print(index_min)

In [ ]:
i = 751
x1_i = x1[i]
x3_i = x3[i]
v2_i = v2[i]
T_i = T[i]

T_exists = T_i>=T_2_limits[0] and T_i<=T_2_limits[1]

if T_exists:
    T_target = T_i
    x1_target = np.interp(T_i, T_2[::-1], x1_2[::-1])
    x3_target = np.interp(x1_target, x1_2[::-1], x3_2[::-1])
    v2_target = np.interp(x1_target, x1_2[::-1], v2_2[::-1])

In [ ]:
(p2.y[0,0]-venus_pos[0])*DU_local

In [ ]:
fig_man, ax_man = plt.subplots(1, subplot_kw={"projection": "3d"})

DU_local = units['DU']/(R_Venus*1000)
p0 = np.array([x1_i, 0, x3_i, 0, v2_i, 0])
p1 = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
              t_span=(0, T[i]),
              y0=p0,
              method='LSODA',
              t_eval = np.linspace(0, T_i, 1000),
              atol=1e-12,
              rtol=1e-12)
ax_man.plot((p1.y[0,:]-venus_pos[0])*DU_local, p1.y[1,:]*DU_local, p1.y[2,:]*DU_local, 'r--', label="L1 северная гало-орбита")
ax_man.plot((p1.y[0,0]-venus_pos[0])*DU_local, p1.y[1,0]*DU_local, p1.y[2,0]*DU_local, 'ro')
traj1 = p1.y[:3, :]   # (3, N)


p0 = np.array([x1_target, 0, -x3_target, 0, v2_target, 0])
p2 = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
              t_span=(0, T[i]),
              y0=p0,
              method='LSODA',
              t_eval = np.linspace(0, T_target, 1000),
              atol=1e-12,
              rtol=1e-12)
ax_man.plot((p2.y[0,:]-venus_pos[0])*DU_local, p2.y[1,:]*DU_local, p2.y[2,:]*DU_local, 'b--', label="L2 южная гало-орбита")
ax_man.plot((p2.y[0,0]-venus_pos[0])*DU_local, p2.y[1,0]*DU_local, p2.y[2,0]*DU_local, 'bo')
ax_man.plot([(p1.y[0,0]-venus_pos[0])*DU_local ,(p2.y[0,0]-venus_pos[0])*DU_local], [p1.y[1,0]*DU_local ,p2.y[1,0]*DU_local], [p1.y[2,0]*DU_local ,p2.y[2,0]*DU_local], 'k--', label="Линия измерений")
traj2 = p2.y[:3, :]   # (3, N)

ax_man.scatter(0, venus_pos[1]*DU_local, venus_pos[2]*DU_local, color='orange', label="Венера")
ax_man.axis('equal')
plt.grid()
plt.legend(loc='upper right')
ax_man.set_xlabel('X, радиусы Венеры,\n ← Солнце')
ax_man.set_ylabel('Y, радиусы Венеры')
ax_man.set_zlabel('Z, радиусы Венеры')
plt.show()

In [ ]:
T_i*TU_local

In [ ]:
A = traj1.T   # (N, 3)
B = traj2.T   # (N, 3)
P = venus_pos # (3,)
AB = B - A
AP = A - P
cross = np.cross(AB, AP)
dists = np.linalg.norm(cross, axis=1) / np.linalg.norm(AB, axis=1)
M = np.mean(dists)
TU_local = units['TU']/(3600*24)
plt.plot(np.linspace(0, T_i*TU_local, 1000),(dists*units['DU']-R_Venus*1000)/1000)
plt.xlabel('Время, сут')
plt.ylabel('Высота измерения, км')
plt.title('Расстояние от линии между траекториями до Венеры')
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.integrate import solve_ivp

# Ваши функции и параметры (примерные)
# def odeCR3BP_3D(t, p, mu):
#     ...

# mu = ...
# x1_i, x3_i, v2_i, T_i = ...
# x1_target, x3_target, v2_target, T_target = ...
# venus_pos = ...

# Интегрируем первую траекторию
p0 = np.array([x1_i, 0, x3_i, 0, v2_i, 0])
t_points = np.linspace(0, T_i, 100)
sol1 = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                 t_span=(0, T_i),
                 y0=p0,
                 method='LSODA',
                 t_eval=t_points,
                 atol=1e-12,
                 rtol=1e-12)

# Интегрируем вторую траекторию
p0_target = np.array([x1_target, 0, x3_target, 0, v2_target, 0])
t_points_target = np.linspace(T_target*0.5, T_target*1.5, 100)
sol2 = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                 t_span=(0, T_target*1.5),
                 y0=p0_target,
                 method='LSODA',
                 t_eval=t_points_target,
                 atol=1e-12,
                 rtol=1e-12)

# Объединяем траектории для анимации (по длине меньшей из них)
min_len = min(sol1.y.shape[1], sol2.y.shape[1])
X1 = sol1.y[:3,:min_len]  # (3,N)
X2 = sol2.y[:3,:min_len]

# Подготовка фигуры
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*venus_pos, color='orange', label="Venus")
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.grid(True)

# Для "equal" осей в 3D:
def set_axes_equal(ax):
    limits = np.array([
        ax.get_xlim3d(),
        ax.get_ylim3d(),
        ax.get_zlim3d(),
    ])
    spans = abs(limits[:, 1] - limits[:, 0])
    centers = np.mean(limits, axis=1)
    radius = 0.1 * max(spans)
    for ctr, ax_set in zip(centers, [ax.set_xlim3d, ax.set_ylim3d, ax.set_zlim3d]):
        ax_set([ctr - radius, ctr + radius])

set_axes_equal(ax)

line1, = ax.plot([], [], [], 'r', label="Halo L1")
line2, = ax.plot([], [], [], 'b', label="Halo L2")
point1, = ax.plot([], [], [], 'ro')
point2, = ax.plot([], [], [], 'bo')
ax.legend()

def init():
    line1.set_data([], [])
    line1.set_3d_properties([])
    line2.set_data([], [])
    line2.set_3d_properties([])
    point1.set_data([], [])
    point1.set_3d_properties([])
    point2.set_data([], [])
    point2.set_3d_properties([])
    return line1, line2, point1, point2

def animate(i):
    # Орбита 1
    line1.set_data(X1[0,:i], X1[1,:i])
    line1.set_3d_properties(X1[2,:i])
    point1.set_data(X1[0,i-1:i], X1[1,i-1:i])
    point1.set_3d_properties(X1[2,i-1:i])
    # Орбита 2
    line2.set_data(X2[0,:i], X2[1,:i])
    line2.set_3d_properties(X2[2,:i])
    point2.set_data(X2[0,i-1:i], X2[1,i-1:i])
    point2.set_3d_properties(X2[2,i-1:i])
    return line1, line2, point1, point2

frames = min_len
anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=frames, interval=20, blit=True)

# Сохраняем гифку (требуется Pillow)
anim.save('halo-orbits.gif', writer='pillow', fps=30)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.integrate import solve_ivp

# --- Ваши функции и параметры ---
# def odeCR3BP_3D(t, p, mu):
#     ...

# mu = ...
# x1_i, x3_i, v2_i, T_i = ...
# x1_target, x3_target, v2_target, T_target = ...
# venus_pos = ...

# Интегрируем первую траекторию (100 точек)
p0 = np.array([x1_i, 0, x3_i, 0, v2_i, 0])
t_points = np.linspace(0, T_i, 100)
sol1 = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                 t_span=(0, T_i),
                 y0=p0,
                 method='LSODA',
                 t_eval=t_points,
                 atol=1e-12,
                 rtol=1e-12)

# Интегрируем вторую траекторию (100 точек)
p0_target = np.array([x1_target, 0, -x3_target, 0, v2_target, 0])
t_points_target = np.linspace(T_target*0, T_target*1, 100)
sol2 = solve_ivp(lambda t, p: odeCR3BP_3D(t, p, mu),
                 t_span=(0, T_target*1.5),
                 y0=p0_target,
                 method='LSODA',
                 t_eval=t_points_target,
                 atol=1e-12,
                 rtol=1e-12)

# Объединяем траектории для анимации (по длине меньшей из них)
min_len = min(sol1.y.shape[1], sol2.y.shape[1])
X1 = sol1.y[:3,:min_len]  # (3,N)
X2 = sol2.y[:3,:min_len]

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*venus_pos, color='orange', label="Venus")
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.grid(True)

def set_axes_equal(ax):
    limits = np.array([
        ax.get_xlim3d(),
        ax.get_ylim3d(),
        ax.get_zlim3d(),
    ])
    spans = abs(limits[:, 1] - limits[:, 0])
    centers = np.mean(limits, axis=1)
    radius = 0.1 * max(spans)
    for ctr, ax_set in zip(centers, [ax.set_xlim3d, ax.set_ylim3d, ax.set_zlim3d]):
        ax_set([ctr - radius, ctr + radius])

set_axes_equal(ax)

# Пунктирные орбиты (отображаются полностью всегда)
# Пунктирные орбиты (отображаются полностью всегда)
orbit1_line, = ax.plot(X1[0], X1[1], X1[2], 'r--', linewidth=1.5, label="Halo L1")
orbit2_line, = ax.plot(X2[0], X2[1], X2[2], 'b--', linewidth=1.5, label="Halo L2")
# Движущиеся точки
point1, = ax.plot([], [], [], 'ro', markersize=7)
point2, = ax.plot([], [], [], 'bo', markersize=7)

ax.legend()

def init():
    point1.set_data([], [])
    point1.set_3d_properties([])
    point2.set_data([], [])
    point2.set_3d_properties([])
    return point1, point2

def animate(i):
    # Перемещаем только точки
    point1.set_data(X1[0,i:i+1], X1[1,i:i+1])
    point1.set_3d_properties(X1[2,i:i+1])
    point2.set_data(X2[0,i:i+1], X2[1,i:i+1])
    point2.set_3d_properties(X2[2,i:i+1])
    return point1, point2

frames = min_len
anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=frames, interval=50, blit=True)

# Сохраняем гифку
anim.save('halo-orbits.gif', writer='pillow', fps=30)

plt.show()